In [ ]:
import io
import pandas as pd
import requests

# URL da API do backend
URL_EXPORTAR = "http://localhost:5000/api/eventos/exportar"

# Obtencao direta dos dados sem necessidade de salvar arquivo intermediario
resposta = requests.get(URL_EXPORTAR)
resposta.raise_for_status()

# Carregamento do CSV em DataFrame
df = pd.read_csv(io.StringIO(resposta.text), sep=";")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["data"] = df["timestamp"].dt.strftime("%Y-%m-%d")

df.head()

In [ ]:
# Parametros para consulta
DIA_ALVO = "2026-10-26"
COLABORADOR_ALVO = "Wagner"

df_dia = df[df["data"] == DIA_ALVO].copy()
print(f"Total de registros para {DIA_ALVO}: {len(df_dia)}")

In [ ]:
total_entradas = (df_dia["tipo_evento"] == "ENTRADA").sum()
total_saidas = (df_dia["tipo_evento"] == "SAIDA").sum()
total_negados = (df_dia["tipo_evento"] == "ACESSO_NEGADO").sum()
total_invasoes = (df_dia["tipo_evento"] == "INVASAO").sum()

print(f"=== METRICAS DO DIA {DIA_ALVO} ===")
print(f"Total de Entradas autorizadas: {total_entradas}")
print(f"Total de Saídas registradas:    {total_saidas}")
print(f"Tentativas de Acesso Negado:   {total_negados}")
print(f"Tentativas de Invasão:         {total_invasoes}")

In [ ]:
# Filtrar apenas eventos de entrada e saida do colaborador no dia
df_colab = df_dia[(df_dia["nome"] == COLABORADOR_ALVO) & (df_dia["tipo_evento"].isin(["ENTRADA", "SAIDA"]))].sort_values("timestamp")

entradas = df_colab[df_colab["tipo_evento"] == "ENTRADA"]["timestamp"].reset_index(drop=True)
saidas = df_colab[df_colab["tipo_evento"] == "SAIDA"]["timestamp"].reset_index(drop=True)

# Pareamento sequencial de cada intervalo entrada -> saida
pares = min(len(entradas), len(saidas))
tempo_total_segundos = sum((saidas[i] - entradas[i]).total_seconds() for i in range(pares))

horas = int(tempo_total_segundos // 3600)
minutos = int((tempo_total_segundos % 3600) // 60)
print(f"Tempo total de permanência de {COLABORADOR_ALVO} no dia {DIA_ALVO}: {horas}h {minutos}min ({tempo_total_segundos:.0f} segundos)")

In [ ]:
# Analise de vulnerabilidade: quais colaboradores cadastrados mais tentaram entrar sem permissao
df_negados = df_dia[df_dia["tipo_evento"] == "ACESSO_NEGADO"]
ranking_negados = df_negados.groupby(["nome", "tag_rfid"]).size().reset_index(name="tentativas").sort_values("tentativas", ascending=False)

print("=== RANKING DE COLABORADORES SEM ACESSO (TENTATIVAS) ===")
ranking_negados